In [28]:
import os, sys
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root not in sys.path:
    sys.path.insert(0, root)

# set random seed for numpy and torch
import random
import numpy as np
import torch

def set_random_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_random_seed(42)

In [29]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from itertools import product
from tqdm import tqdm
from copy import deepcopy

from classifier.models import PeptidePredictor
from cvae.models import CVAESimpleEnc
from cvae.utils import (
    CONDITION_LENGTH, MAX_SEQ_LENGTH, PAD_TOKEN_ID, ALPHABET, BOS_ID, EOS_ID,
    esm_model_pretrained, idx_to_fasta, convert_and_pad,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
print(torch.cuda.device_count())

cuda
1


In [30]:
peptide_predictor = PeptidePredictor(deepcopy(esm_model_pretrained), alphabet=ALPHABET)

peptide_predictor_dict = torch.load("peptide_predictor.pt", map_location=device, weights_only=True)
peptide_predictor.load_state_dict(peptide_predictor_dict)

<All keys matched successfully>

In [31]:
peptide_predictor.eval()

peptide_predictor.to(device)
print("Peptide Predictor initialized")

Peptide Predictor initialized


In [32]:
# generate 4800 random sequences, roughly uniformly distributed across lengths 4-10

seqs = []
for _ in range(4800):
    length = np.random.randint(4, 11)
    seq = np.random.choice(list('ACDEFGHIKLMNPQRSTVWY'), size=length)
    seqs.append(''.join(seq))

In [33]:
# classify sequences
batch_size = 64

batch_data = [(seq, len(seq)) for seq in seqs]
results = []
min_ap, max_ap = 0.959986, 2.89703

for start_idx in tqdm(range(0, len(batch_data), batch_size), desc="Classifying batches"):
        chunk = batch_data[start_idx:start_idx + batch_size]
        data = [(f"peptide_{start_idx + i}", seq) for i, (seq, _,) in enumerate(chunk)]
        tokens = convert_and_pad(data, seq_length=MAX_SEQ_LENGTH).to(device)

        with torch.no_grad():
            ap_preds, cls_preds = peptide_predictor(tokens)

        for (seq, length), ap_pred, cls_pred in zip(chunk, ap_preds, cls_preds):
            ap_val = ap_pred.item() * (max_ap - min_ap) + min_ap
            cls_prob = torch.sigmoid(cls_pred).item()
            results.append((seq, length, ap_val, cls_prob))

Classifying batches:   0%|          | 0/75 [00:00<?, ?it/s]

Classifying batches: 100%|██████████| 75/75 [00:08<00:00,  9.08it/s]


In [34]:
# convert sequences and predictions to a dataframe
results_df = pd.DataFrame(results, columns=['sequence', 'length', 'ap', 'clf'])

results_df 

,sequence,length,ap,clf
0,YRMIHWMMEI,10,1.957391,0.993672
1,CNGCAN,6,1.473979,0.005742
2,NTLSR,5,1.628775,0.258079
3,RWNYDFWHK,9,1.805670,0.998484
4,VEQVKCYRHN,10,1.700657,0.321653
...,...,...,...,...
4795,QCAPIKYC,8,1.732509,0.677600
4796,GDYT,4,1.607059,0.153480
4797,IQLYLRIMS,9,1.833378,0.971954
4798,DRPMQ,5,1.372599,0.002404


In [37]:
# remove all sequences with length=4
results_df = results_df[results_df['length'] > 4]
results_df

,sequence,length,ap,clf
0,YRMIHWMMEI,10,1.957391,0.993672
1,CNGCAN,6,1.473979,0.005742
2,NTLSR,5,1.628775,0.258079
3,RWNYDFWHK,9,1.805670,0.998484
4,VEQVKCYRHN,10,1.700657,0.321653
...,...,...,...,...
4794,FKWAIYETAC,10,1.894147,0.993665
4795,QCAPIKYC,8,1.732509,0.677600
4797,IQLYLRIMS,9,1.833378,0.971954
4798,DRPMQ,5,1.372599,0.002404


In [40]:
# add more random with length=5-10 until it has 4800 sequences
new_seqs = []
while len(results_df) + len(new_seqs) < 4800:
    length = np.random.randint(5, 11)
    seq = np.random.choice(list('ACDEFGHIKLMNPQRSTVWY'), size=length)
    new_seqs.append(''.join(seq))

In [41]:
new_seqs_data = [(seq, len(seq)) for seq in new_seqs]
new_results = []
for start_idx in tqdm(range(0, len(new_seqs_data), batch_size), desc="Classifying new sequences"):
    chunk = new_seqs_data[start_idx:start_idx + batch_size]
    data = [(f"peptide_new_{start_idx + i}", seq) for i, (seq, _,) in enumerate(chunk)]
    tokens = convert_and_pad(data, seq_length=MAX_SEQ_LENGTH).to(device)

    with torch.no_grad():
        ap_preds, cls_preds = peptide_predictor(tokens)

    for (seq, length), ap_pred, cls_pred in zip(chunk, ap_preds, cls_preds):
        ap_val = ap_pred.item() * (max_ap - min_ap) + min_ap
        cls_prob = torch.sigmoid(cls_pred).item()
        new_results.append((seq, length, ap_val, cls_prob))

new_results_df = pd.DataFrame(new_results, columns=['sequence', 'length', 'ap', 'clf'])
new_results_df

Classifying new sequences:   0%|          | 0/11 [00:00<?, ?it/s]

Classifying new sequences: 100%|██████████| 11/11 [00:01<00:00, 10.31it/s]


,sequence,length,ap,clf
0,FHVVSHCTI,9,1.939296,0.999871
1,VWYKNW,6,2.020619,0.999872
2,IILVRMQRQN,10,1.445622,0.003782
3,DNHHWQAH,8,1.688305,0.447150
4,LVPIK,5,1.681138,0.604553
...,...,...,...,...
650,PVCAKVFDF,9,1.904546,0.990907
651,IIHCAAIT,8,1.825752,0.917744
652,QIYVHQQGD,9,1.652297,0.132632
653,QQDLKHHHN,9,1.742841,0.183048


In [42]:
# save generated fibers to file
with open("generated_random_init.txt", "w") as f:
    for seq in results_df['sequence']:
        f.write(f"{seq}\n")
    for seq in new_results_df['sequence']:
        f.write(f"{seq}\n")

In [43]:
# filter peptides with predicted ap > 1.8
filtered_df = results_df[results_df['ap'] > 1.8]
filtered_df

,sequence,length,ap,clf
0,YRMIHWMMEI,10,1.957391,0.993672
3,RWNYDFWHK,9,1.805670,0.998484
15,LPWHT,5,2.068440,0.999762
17,HWCLPGNN,8,1.878374,0.972086
19,HGIKFA,6,1.852045,0.979549
...,...,...,...,...
4787,IQPTH,5,1.831814,0.919795
4788,KYYIQYDPK,9,1.828462,0.977130
4790,YIHIKADVW,9,1.932356,0.994987
4794,FKWAIYETAC,10,1.894147,0.993665


In [44]:
# filtered new_seqs
filtered_new_df = new_results_df[new_results_df['ap'] > 1.8]
filtered_new_df

,sequence,length,ap,clf
0,FHVVSHCTI,9,1.939296,0.999871
1,VWYKNW,6,2.020619,0.999872
7,KHPTP,5,1.873732,0.974508
12,TWFEVI,6,2.028403,0.999775
14,KPYIMKVYD,9,1.871903,0.980221
...,...,...,...,...
643,YMWGLRVHDS,10,1.878590,0.992765
645,FCMPFIM,7,2.151571,0.999898
648,TTWPWTYNY,9,1.893596,0.999984
650,PVCAKVFDF,9,1.904546,0.990907


In [45]:
with open("filtered_ap_peptides_random.txt", "w") as f:
    # save sequence, length, ap, clf
    for idx, row in filtered_df.iterrows():
        f.write(f"{row['sequence']},{row['length']},{row['ap']},{row['clf']}\n")
    for idx, row in filtered_new_df.iterrows():
        f.write(f"{row['sequence']},{row['length']},{row['ap']},{row['clf']}\n")

In [27]:
# save new seqs to fst
new_filtered_seqs = filtered_new_df['sequence'].tolist()
with open("random_peptides_filtered_extra.fst", "w") as f:
    for i, seq in enumerate(new_filtered_seqs):
        f.write(f">random_peptide_{i}\n{seq}\n")

In [ ]:
# get generated sequences
filtered_seqs = filtered_df['sequence'].tolist()
with open("random_peptides_filtered.fst", "w") as f:
    for i, seq in enumerate(filtered_seqs):
        f.write(f">peptide_{i+1}\n")
        f.write(f"{seq}\n")


In [30]:
columns = ['length', 'ap', 'is_assembled',
                'hydrophobic_moment', 'has_beta_sheet_content',
                'net_charge', 
                ]

In [31]:
# load descriptors

desc_df = pd.read_csv("random_peptides_descriptors.csv")
merged_df = pd.merge(filtered_df, desc_df, on='sequence')
merged_df['is_assembled'] = merged_df['ap'] >= 1.8
merged_df['has_beta_sheet_content'] = merged_df['beta_sheet_fraction'] > 0.0
final_df = merged_df[columns]
final_df 

,length,ap,is_assembled,hydrophobic_moment,has_beta_sheet_content,net_charge
0,10,1.957391,True,0.208000,False,0
1,9,1.805670,True,0.337778,False,1
2,5,2.068440,True,0.308000,False,0
3,8,1.878374,True,0.100000,False,0
4,6,1.852045,True,0.295000,False,1
...,...,...,...,...,...,...
1606,5,1.831814,True,0.040000,False,0
1607,9,1.828462,True,0.274444,False,1
1608,9,1.932356,True,0.303333,False,0
1609,10,1.894147,True,0.288000,False,0


In [32]:
min_hydro, max_hydro = 0, 1.99800
min_charge, max_charge = -6.0, 6.0

final_df['hydrophobic_moment'] = (final_df['hydrophobic_moment'] - min_hydro) / (max_hydro - min_hydro)
final_df['net_charge'] = (final_df['net_charge'] - min_charge) / (max_charge - min_charge)

final_df

/tmp/ipykernel_58760/641255534.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['hydrophobic_moment'] = (final_df['hydrophobic_moment'] - min_hydro) / (max_hydro - min_hydro)
/tmp/ipykernel_58760/641255534.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['net_charge'] = (final_df['net_charge'] - min_charge) / (max_charge - min_charge)


,length,ap,is_assembled,hydrophobic_moment,has_beta_sheet_content,net_charge
0,10,1.957391,True,0.104104,False,0.500000
1,9,1.805670,True,0.169058,False,0.583333
2,5,2.068440,True,0.154154,False,0.500000
3,8,1.878374,True,0.050050,False,0.500000
4,6,1.852045,True,0.147648,False,0.583333
...,...,...,...,...,...,...
1606,5,1.831814,True,0.020020,False,0.500000
1607,9,1.828462,True,0.137360,False,0.583333
1608,9,1.932356,True,0.151818,False,0.500000
1609,10,1.894147,True,0.144144,False,0.500000


In [36]:
fiber_ranges = {
    'length': (7, 11, 1),
    'is_assembled': (1, 2, 1),
    'has_beta_sheet_content': (1, 2, 1),
    'net_charge': (0.4, 0.6, 0.05),
}

sphere_ranges = {
    'length': (4, 8, 1),
    'is_assembled': (1, 2, 1),
    'hydrophobic_moment': (0.6, 1.05, 0.1),
    'net_charge': (0.4, 0.6, 0.05),
}

In [54]:
# get peptides with desired fiber properties
fiber_filtered = final_df[
    (final_df['length'] >= fiber_ranges['length'][0]) & (final_df['length'] < fiber_ranges['length'][1]) &
    (final_df['is_assembled'] == 1) & (final_df['has_beta_sheet_content'] == 1) & 
    (final_df['net_charge'] >= fiber_ranges['net_charge'][0]) & (final_df['net_charge'] < fiber_ranges['net_charge'][1])
]

fiber_filtered

,length,ap,is_assembled,hydrophobic_moment,has_beta_sheet_content,net_charge
187,10,1.842635,True,0.077077,True,0.500000
599,10,1.854066,True,0.110611,True,0.416667
731,10,1.837791,True,0.302302,True,0.583333
1159,10,1.862551,True,0.081081,True,0.416667


In [55]:
# get peptides with desired sphere properties
sphere_filtered = final_df[
    (final_df['length'] >= sphere_ranges['length'][0]) & (final_df['length'] < sphere_ranges['length'][1]) &
    (final_df['is_assembled'] == 1) & 
    (final_df['hydrophobic_moment'] >= sphere_ranges['hydrophobic_moment'][0]) & (final_df['hydrophobic_moment'] < sphere_ranges['hydrophobic_moment'][1]) &
    (final_df['net_charge'] >= sphere_ranges['net_charge'][0]) & (final_df['net_charge'] < sphere_ranges['net_charge'][1])
]

sphere_filtered

,length,ap,is_assembled,hydrophobic_moment,has_beta_sheet_content,net_charge
